# 📈 Forecastability Profile Walkthrough — F1

**Repo:** [AdamKrysztopa/dependence-forecastability](https://github.com/AdamKrysztopa/dependence-forecastability) &nbsp;·&nbsp; **Paper:** [arXiv:2603.27074](https://arxiv.org/abs/2603.27074) (Catt, 2026)

This notebook explains the **F1 Forecastability Profile** — the horizon-wise map

$$F(h;\,\mathcal{I}_t) = I(Y_{t+h};\,\mathcal{I}_t)$$

computed by the triage pipeline.  It shows:

1. What a **forecastability profile** is vs a single scalar score
2. How the **informative horizon set** $\mathcal{H}_\varepsilon = \{h : F(h) \ge \varepsilon\}$ is derived
3. Contrast between a **non-monotone** (seasonal) profile and a **monotone-decaying** (AR) profile
4. How to read the actionable **modelling recommendations**

## Setup ⚙️

```bash
uv sync --group notebook
uv run python -m ipykernel install --user --name forecastability
```

In [ ]:
%matplotlib inline
import os
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("MPLBACKEND", "inline")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

mpl.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Environment ready ✓")

In [ ]:
# ── F1 imports ───────────────────────────────────────────────────────────────
from forecastability.utils.datasets import generate_ar1, load_air_passengers
from forecastability.triage.forecastability_profile import ForecastabilityProfile
from forecastability.triage.models import TriageRequest
from forecastability.use_cases.run_triage import run_triage

print("Imports OK ✓")

---

## 1 · Concept — Profile vs Scalar

Classical forecastability approaches collapse information across all horizons into a single number.  The **forecastability profile** keeps the full horizon-wise picture:

| | Scalar summary | Forecastability profile |
|---|---|---|
| Horizon resolution | One number | One value per lag $h$ |
| Detects seasonal reversal | ✗ | ✓ |
| Actionable horizon recommendations | ✗ | ✓ |
| Informative horizon set | ✗ | ✓ |

### The equations

For forecast target $Y_{t+h}$ and information set $\mathcal{I}_t$:

$$F(h;\,\mathcal{I}_t) = I(Y_{t+h};\,\mathcal{I}_t) = H(Y_{t+h}) - H(Y_{t+h} \mid \mathcal{I}_t)$$

The **informative horizon set** is defined by a threshold $\varepsilon$ (derived from surrogate significance testing):

$$\mathcal{H}_\varepsilon = \bigl\{h : F(h;\,\mathcal{I}_t) \ge \varepsilon\bigr\}$$

> The profile need not be monotonically decreasing!  Seasonal processes produce
> characteristic bumps at multiples of the dominant period.

---

## 2 · Signal generation

We use two synthetic signals:

- **Seasonal (non-monotone)** — superposition of two sinusoids with different periods, amplitude modulation, and Gaussian noise.  Expected to produce a non-monotone profile with a bump near the dominant period.
- **AR(1) (monotone)** — autoregressive process with strong memory ($\phi = 0.85$).  Expected to produce a smoothly decaying profile.

In [ ]:
RNG = np.random.default_rng(42)
N = 720

# ── Non-monotone seasonal signal ─────────────────────────────────────────────
t = np.arange(N, dtype=float)
primary   = np.sin(2.0 * np.pi * t / 12.0)
secondary = 0.55 * np.sin(2.0 * np.pi * t / 6.0 + 0.45)
envelope  = 1.0 + 0.20 * np.sin(2.0 * np.pi * t / 72.0)
noise_s   = RNG.normal(0.0, 0.20, size=N)
series_seasonal = envelope * (primary + secondary) + noise_s

# ── AR(1) monotone signal ─────────────────────────────────────────────────────
series_ar = generate_ar1(n_samples=N, phi=0.85, random_state=42)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(series_seasonal, lw=0.9, color="steelblue")
axes[0].set_title("Seasonal signal (non-monotone profile expected)")
axes[1].plot(series_ar, lw=0.9, color="darkorange")
axes[1].set_title("AR(1) signal (monotone-decaying profile expected)")
axes[1].set_xlabel("Time index")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

---

## 3 · Run triage and extract F1 profiles

In [ ]:
MAX_LAG      = 24
N_SURROGATES = 99
RANDOM_STATE = 42

def run_and_extract_profile(series: np.ndarray, label: str) -> ForecastabilityProfile:
    req    = TriageRequest(series=series, max_lag=MAX_LAG,
                           n_surrogates=N_SURROGATES, random_state=RANDOM_STATE)
    result = run_triage(req)
    if result.blocked or result.forecastability_profile is None:
        raise RuntimeError(f"Triage blocked or no profile for '{label}'")
    return result.forecastability_profile

print("Running triage for seasonal signal...")
profile_seasonal = run_and_extract_profile(series_seasonal, "seasonal")
print("Running triage for AR(1) signal...")
profile_ar       = run_and_extract_profile(series_ar, "ar1")
print("Done ✓")

---

## 4 · Visualise and compare profiles

The key visual features:

- **Red markers** — horizons that belong to the informative set $\mathcal{H}_\varepsilon$
- **Orange dashed line** — threshold $\varepsilon$ (derived from surrogate tests)
- **Non-monotone** profile shows bumps at multiples of the seasonal period
- **Monotone** profile decays smoothly from lag 1

In [ ]:
def plot_profile(ax: plt.Axes, profile: ForecastabilityProfile, title: str, color: str) -> None:
    horizons   = np.asarray(profile.horizons)
    values     = profile.values
    informative = set(profile.informative_horizons)

    ax.plot(horizons, values, marker="o", ms=4, lw=1.8, color=color, label="F(h)")
    ax.axhline(profile.epsilon, ls="--", lw=1.2, color="darkorange", label=f"ε = {profile.epsilon:.4f}")

    if informative:
        mask = np.array([h in informative for h in horizons])
        ax.scatter(horizons[mask], values[mask], color="crimson", s=55, zorder=4,
                   label="informative horizon")

    if profile.peak_horizon:
        ax.axvline(profile.peak_horizon, ls=":", lw=1, color="gray", alpha=0.7,
                   label=f"peak h={profile.peak_horizon}")

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Horizon h")
    ax.set_ylabel("F(h) — AMI")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
plot_profile(ax1, profile_seasonal, "Seasonal (non-monotone)",  "steelblue")
plot_profile(ax2, profile_ar,       "AR(1)   (monotone decay)", "darkorange")
fig.suptitle("F1 Forecastability Profiles — Profile Shape Comparison",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

---

## 5 · Inspect profile fields

In [ ]:
def print_profile(label: str, p: ForecastabilityProfile) -> None:
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  horizons       : {p.horizons}")
    print(f"  values (round) : {np.round(p.values, 4)}")
    print(f"  epsilon        : {p.epsilon:.6f}")
    print(f"  informative    : {p.informative_horizons}")
    print(f"  peak_horizon   : {p.peak_horizon}")
    print(f"  is_non_monotone: {p.is_non_monotone}")
    print(f"  summary        : {p.summary}")
    print(f"  model_now      : {p.model_now}")
    print(f"  review_horizons: {p.review_horizons}")
    print(f"  avoid_horizons : {p.avoid_horizons}")

print_profile("Seasonal — non-monotone profile", profile_seasonal)
print_profile("AR(1)    — monotone profile",     profile_ar)

---

## 6 · Profile on a real dataset — AirPassengers

The classic AirPassengers series has a strong monthly seasonal cycle. We expect a
non-monotone profile with peaks near horizon 12 and possibly 6 (the sub-harmonic).

In [ ]:
series_ap = load_air_passengers().astype(float)

print(f"AirPassengers: n={len(series_ap)}, range=[{series_ap.min():.0f}, {series_ap.max():.0f}]")

profile_ap = run_and_extract_profile(series_ap, "AirPassengers")

fig, (ax_s, ax_p) = plt.subplots(1, 2, figsize=(14, 5))

ax_s.plot(series_ap, lw=1.3, color="teal")
ax_s.set_title("AirPassengers series (Jan 1949 – Dec 1960)")
ax_s.set_xlabel("Month index")
ax_s.set_ylabel("Passengers (thousands)")
ax_s.grid(alpha=0.25)

plot_profile(ax_p, profile_ap, "AirPassengers — forecastability profile", "teal")

fig.tight_layout()
plt.show()
print_profile("AirPassengers", profile_ap)

---

## 7 · Scalar vs profile — the information cost of collapsing to a single number

A single AUC value (trapezoidal integral) captures *total area* but loses the
horizon-specific structure needed for modelling decisions.

In [ ]:
def profile_auc(p: ForecastabilityProfile) -> float:
    return float(np.trapezoid(p.values, np.asarray(p.horizons)))

results = {
    "Seasonal (non-monotone)": profile_seasonal,
    "AR(1) (monotone)":        profile_ar,
    "AirPassengers":           profile_ap,
}

rows = []
for label, p in results.items():
    rows.append({
        "signal":            label,
        "AUC (scalar)": round(profile_auc(p), 4),
        "peak horizon":      p.peak_horizon,
        "# informative":     len(p.informative_horizons),
        "non-monotone":      p.is_non_monotone,
        "model_now": p.model_now.split("—")[0].strip(),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

---

## 8 · Modelling recommendation walkthrough

The service maps the profile shape to an actionable recommendation:

| `model_now` level | Condition |
|---|---|
| `HIGH` | ≥1 informative horizon, peak MI > 0.15 |
| `MEDIUM` | ≥1 informative horizon, 0.05 < peak MI ≤ 0.15 |
| `LOW` | ≥1 informative horizon, peak MI ≤ 0.05 |
| `NONE` | No informative horizons |

The `review_horizons` list is the **informative horizon set** $\mathcal{H}_\varepsilon$.
The `avoid_horizons` list is the complement — horizons where MI is below $\varepsilon$,
which should **not** be included in model feature windows.

In [ ]:
for label, p in results.items():
    print(f"\n{label}")
    print(f"  model_now      : {p.model_now}")
    print(f"  review_horizons: {p.review_horizons}")
    print(f"  avoid_horizons : {p.avoid_horizons}")

---

## 9 · Save profile comparison figure

We save a final side-by-side comparison of all three profiles for reporting.

In [ ]:
from pathlib import Path

OUT_DIR = Path("../../outputs/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

colors = ["steelblue", "darkorange", "teal"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, (label, p), color in zip(axes, results.items(), colors):
    plot_profile(ax, p, label, color)

fig.suptitle("F1 Forecastability Profile — All Signals",
             fontsize=13, fontweight="bold")
fig.tight_layout()

out_path = OUT_DIR / "f1_forecastability_profile_comparison.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved → {out_path}")
plt.show()

---

## Summary

| Take-away | Detail |
|---|---|
| Profile is horizon-wise MI | $F(h; \mathcal{I}_t) = I(Y_{t+h}; \mathcal{I}_t)$ — one value per lag |
| Not necessarily monotone | Seasonal processes produce bumps at multiples of the period |
| Informative horizon set | $\mathcal{H}_\varepsilon$ derived from surrogate significance testing |
| Scalar AUC loses structure | Two signals can share the same AUC yet have very different profile shapes |
| Actionable output | `model_now`, `review_horizons`, `avoid_horizons` tell you what to do |

**Next notebook →** `02_information_limits_and_compression.ipynb` — theoretical ceilings,
exploitation ratios, and what compression does to the ceiling.